# 第13回：Kaggleに入って最初の提出を作る

**今日の問い：コンペの説明を、ローカルの分析手順へどう翻訳するか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 問題・指標・データ・提出形式を読み解く
- 再現可能なベースラインをローカル評価する
- 提出CSVを機械的に検査する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- Leaderboard：提出結果を順位表示する仕組み
- Public/Private：公開中と最終判定で使う評価データの区分
- submission：指定形式の予測ファイル
- ベースライン：最初に必ず保存する比較起点

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
sample = pd.read_csv(DATA / "local_competition" / "sample_submission.csv")
print("train:", train.shape, "test:", test.shape, "提出見本:", sample.shape)
display(train.head(3))
display(sample.head(3))


## コンペ説明

- 目的：実験計画時の情報から活性`active`（0/1）を予測する
- 指標：F1
- `train.csv`には答えがある
- `test.csv`には答えがない
- 提出列は`sample_id`と`active`

Kaggle Titanicを使える場合も、最初に同じ4点を確認します。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

target="active"
drop_columns=["sample_id", "experiment_date", "smiles", target]
features=[column for column in train.columns if column not in drop_columns]
numeric=train[features].select_dtypes(include="number").columns.tolist()
categorical=[column for column in features if column not in numeric]
preprocess=ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
model=Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42))])
X_train, X_valid, y_train, y_valid=train_test_split(train[features], train[target], test_size=0.25, random_state=42, stratify=train[target])
model.fit(X_train, y_train)
print("ローカル検証F1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


## TRY：提出CSVを作り、機械的に検査する


In [ ]:
model.fit(train[features], train[target])
submission=pd.DataFrame({"sample_id": test["sample_id"], "active": model.predict(test[features])})
assert list(submission.columns) == ["sample_id", "active"]
assert len(submission) == len(test)
assert submission["sample_id"].is_unique
output=ROOT / "workspace" / "submission_baseline.csv"
submission.to_csv(output, index=False)
print("保存先:", output)
submission.head()


## CHANGE

提出前に変えるのは1点だけです。例：`max_depth=6`を`3`へ変え、ローカル検証がどう変わるか確認します。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- testには答えがないことを確認する
- ローカル検証とLeaderboardの役割を分ける
- ID列の順序と一意性を検査する


In [ ]:
def validate_submission(submission, test, expected_columns):
    assert list(submission.columns) == expected_columns, "列名または順序が違います"
    assert len(submission) == len(test), "行数がtestと一致しません"
    assert submission[expected_columns[0]].is_unique, "IDが重複しています"
    assert submission[expected_columns[0]].tolist() == test[expected_columns[0]].tolist(), "IDの順序がtestと一致しません"
    assert submission[expected_columns[1]].isin([0, 1]).all(), "予測値は0/1にしてください"
    return "提出形式OK"

print(validate_submission(submission, test, ["sample_id", "active"]))
display(train[target].value_counts(normalize=True).rename("割合").to_frame().round(3))


## よくある誤り

- testの情報へ合わせて特徴量を決める
- 提出ファイルのindex列を混入させる
- 最初から公開Notebookを丸ごと写す

## SELF-STUDY（任意・30〜60分）

- データ辞書を自分の言葉で1ページにする
- ベースライン提出後に変更点を1つだけ試す

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 評価指標は何か
2. trainとtestの違いは何か
3. 提出前に検査する3項目は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
